# 04 · Downstream tasks: default in the next 12 months

**Owner:** Aditya · walkthrough notebook (no exercises)

A foundation model is trained on events with no labels. A **downstream task** is a real business
question we answer with it. In this notebook the question is:

> Will this customer default in the 12 months after the cut date?

We go through the three steps every downstream task needs:

1. **Training data:** one row per customer with a label.
2. **Embeddings:** one vector per customer from the foundation model (fm_t4).
3. **Models on the embeddings:** logistic regression (LR), gradient boosted trees (GBDT) and a
   small neural network (MLP), and how to judge them.

**Data:** two customer sets from the bank data. `out_v4_2025` is the train set and `out_v5_2025`
is the test set. They have no customers in common.

**Runs on:** the VM, with an environment that has `pragmatiq`, `torch`, `scikit-learn` and `pandas`
(for example `~/pragma`). Section 2 needs a GPU; it also works on CPU, only slower.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.compute as pc
import pyarrow.dataset as pds
import pyarrow.parquet as pq
import torch
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from pragmatiq import api
from pragmatiq.data.sharding import record_from_row
from pragmatiq.data.tokenizer import PragmaTokenizer

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

# ---- paths (all on the VM) ----
ROOT = Path("/home/shared/foundation-model-prod")
PREP = Path("/home/shared/shared_aditya")

RAW = {"train": ROOT / "out_v4_2025", "test": ROOT / "out_v5_2025"}
LABELS = {"train": ROOT / "out_v4_2025_labels/default_12m_all.parquet",
          "test": ROOT / "out_v5_2025_labels/default_12m_all.parquet"}
INTAKE_TEST = PREP / "prepared_out_v5_2025/intake"            # intake output for the test set
EMBEDDINGS = {"train": PREP / "prepared_out_v4_2025/embeddings.parquet",
              "test": PREP / "prepared_out_v5_2025/embeddings.parquet"}

FM_RUN = ROOT / "work_t4/runs/fm_t4"                          # the foundation model
TOKENIZER = FM_RUN / "tokenizer"                              # its tokenizer

# your own scratch folder for the live demo in section 2
DEMO = Path.home() / "downstream_demo"
DEMO.mkdir(exist_ok=True)
N_DEMO = 500

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
for p in [*RAW.values(), *LABELS.values(), INTAKE_TEST, *EMBEDDINGS.values(), FM_RUN, TOKENIZER]:
    print(("ok      " if p.exists() else "MISSING ") + str(p))

---
## 1. Making the training data (labels)

A downstream task needs one row per customer:

| column | meaning |
|---|---|
| `user_id` | the customer |
| `eval_ts` | the **cut date**: the moment we make the prediction |
| `label` | 1 if the customer defaults in the 12 months after `eval_ts`, else 0 |

**Default** here means: any of the customer's products reaches **90 or more days past due**.

Two rules make the labels usable:

- The model only sees data from **before** the cut date. The label comes from **after** it.
  If data from after the cut gets into the model, the test results look better than they will
  be in real use. This is called **leakage**.
- The train and test sets are **different customers**, so the test shows how the model does on
  customers it has never seen.

The label files are already made. Let's load them.

In [ ]:
labels = {k: pd.read_parquet(p) for k, p in LABELS.items()}
display(labels["train"].head())

for k, df in labels.items():
    print(f"{k:5}: {len(df):6,} customers, {int(df.label.sum()):3} defaulters "
          f"({df.label.mean():.2%}), cut date {sorted(df.eval_ts.unique())}")

overlap = set(labels["train"].user_id) & set(labels["test"].user_id)
print("customers in both train and test:", len(overlap))

**Things to notice:**

- Fewer than 1 in 200 customers default. Most of the work in section 3 is about dealing with a
  class this rare.
- The test set has only 42 defaulters. Every result we get will have a wide range of
  uncertainty. We will measure that range.

### Where a label comes from

Let's check the raw data for one defaulter and one non-defaulter. We read `product_snapshot`,
which records how many days each product is past due, and look at the 12 months after the cut.

In [ ]:
def product_history(folder, customer_id):
    """All product_snapshot rows for one customer."""
    ds = pds.dataset(str(folder / "product_snapshot"), format="parquet")
    t = ds.to_table(filter=pc.field("customer_id") == customer_id,
                    columns=["customer_id", "snapshot_date", "days_delinquent"])
    df = t.to_pandas()
    df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])
    return df.sort_values("snapshot_date")


test_labels = labels["test"]
cut = pd.Timestamp(test_labels.eval_ts.iloc[0])
window_end = cut + pd.DateOffset(months=12)

for lab in (1, 0):
    cid = test_labels[test_labels.label == lab].user_id.iloc[0]
    h = product_history(RAW["test"], cid)
    before = h[h.snapshot_date < cut]
    after = h[(h.snapshot_date >= cut) & (h.snapshot_date < window_end)]
    print(f"\ncustomer {cid}, label {lab}")
    print(f"  most days past due before the cut : {before.days_delinquent.max()}")
    print(f"  most days past due in the window  : {after.days_delinquent.max()}")
    print(f"  first snapshot with 90+ days in the window: "
          f"{after[after.days_delinquent >= 90].snapshot_date.min()}")

The defaulter reaches 90+ days past due inside the 12-month window. The non-defaulter does not.
That is all a label is: a fact about the future, computed from the raw tables, attached to a
customer at the cut date.

---
## 2. Making embeddings

The foundation model turns a customer's whole history into **one vector of 192 numbers**. This
vector is the customer's **embedding**. Models in section 3 use it instead of the raw tables.

There are three steps:

| step | input | output | tool |
|---|---|---|---|
| intake | the bank's 12 raw tables | one event list per customer + a profile | `intake` |
| tokenize | event lists | token ids, using the **same tokenizer** the model was trained with | `pragmatiq tokenize` |
| embed | token ids | one 192-number vector per customer | `pragmatiq embed` |

The intake for the test set is already done (it reads the whole folder, which takes too long for
a live session). We run tokenize and embed live on **500 test customers**.

### 2.1 Intake output: one customer as events

In [ ]:
events = pq.read_table(INTAKE_TEST / "events.parquet")
profiles = pq.read_table(INTAKE_TEST / "profiles.parquet")
print(f"{events.num_rows:,} events, {profiles.num_rows:,} customers")
print(events.schema)

one_id = events.column("user_id")[0].as_py()
mine = events.filter(pc.equal(events.column("user_id"), one_id)).to_pandas()
print(f"\ncustomer {one_id}: {len(mine):,} events")
display(mine.source.value_counts().to_frame("events"))
display(mine.head(5))

Every row is one event: **when** (`ts`), **what kind** (`source`) and **the details** (`fields`,
a list of key–value pairs). All 12 tables are now in this one shape. The model never sees the
tables themselves.

In [ ]:
row = mine.iloc[0]
print(row.ts, row.source)
for k, v in row.fields:
    print(f"    {k:30} {v}")

prof = profiles.filter(pc.equal(profiles.column("user_id"), one_id)).to_pylist()[0]
print("\nprofile attributes (first 10):")
for k, v in list(prof["attributes"])[:10]:
    print(f"    {k:30} {v}")

### 2.2 Tokenize 500 customers

The tokenizer turns each key and value into an integer id. We **must** use the tokenizer that
came with fm_t4. A different tokenizer gives different ids, and the model would read nonsense.
`pragmatiq embed` checks this and stops if they don't match.

`max_users=500` takes the first 500 customers in the intake file.

In [ ]:
tok_dir = DEMO / "tokenized"
manifest = api.tokenize(data_dir=INTAKE_TEST, out=tok_dir, tokenizer_dir=TOKENIZER,
                        max_users=N_DEMO)
print({k: manifest[k] for k in ("n_users", "total_events", "total_tokens") if k in manifest})

Let's look at one customer's first event as token ids, and then turn the ids back into words.
Numbers come back as a range (a "bucket"), because the tokenizer groups numbers into 64 buckets.

In [ ]:
tok = PragmaTokenizer.load(tok_dir / "tokenizer")
shard = sorted((tok_dir / "shards").glob("*.parquet"))[0]
rec = record_from_row(pq.read_table(shard).slice(0, 1).to_pylist()[0])

print(f"customer {rec.user_id}: {rec.n_events:,} events, {rec.n_tokens:,} tokens")
lo, hi = int(rec.event_offsets[0]), int(rec.event_offsets[1])
print("\nfirst event as ids:")
print("  key ids  :", rec.key_ids[lo:hi].tolist())
print("  value ids:", rec.value_ids[lo:hi].tolist())
print("\nfirst event decoded:")
for k, v in tok.decode_event(rec, 0):
    print(f"    {k:30} {v}")

### 2.3 Embed the 500 customers

In [ ]:
demo_emb_path = DEMO / "embeddings_500.parquet"
info = api.embed(tok_dir, FM_RUN, out=demo_emb_path, device=DEVICE)
print(info)

demo_emb = pd.read_parquet(demo_emb_path)
display(demo_emb.head())
demo_x = np.stack(demo_emb.embedding.to_numpy())
print("shape:", demo_x.shape, "-> one row per customer, 192 numbers each")

**Check:** the same customers are in the full embedding file made earlier for the whole test
set. Our 500 vectors should match them. On a GPU the model runs in bf16, so the numbers can differ
in the last digits. We compare with **cosine similarity** (1.0 = same direction).

In [ ]:
full_test = pd.read_parquet(EMBEDDINGS["test"]).set_index("user_id")
ref = np.stack(full_test.loc[demo_emb.user_id].embedding.to_numpy())
cos = (demo_x * ref).sum(1) / (np.linalg.norm(demo_x, axis=1) * np.linalg.norm(ref, axis=1))
print(f"cosine similarity, ours vs precomputed: min {cos.min():.4f}, mean {cos.mean():.4f}")

From here on we use the **precomputed** embeddings for all customers: about 18,000 train and
9,000 test. They were made with exactly these steps (`kumo/prep_new.sh`).

---
## 3. Models on the embeddings

### 3.1 Join labels and embeddings

Each customer gets one row: 192 embedding numbers (`X`) and the label (`y`). We match by
`user_id`, never by row order.

In [ ]:
def load_xy(split):
    emb = pd.read_parquet(EMBEDDINGS[split])
    df = labels[split].merge(emb, on="user_id", how="inner")
    print(f"{split:5}: {len(labels[split]):,} labelled, {len(df):,} with an embedding, "
          f"{int(df.label.sum())} defaulters")
    return np.stack(df.embedding.to_numpy()).astype("float32"), df.label.to_numpy(), df


x_tr, y_tr, train_df = load_xy("train")
x_te, y_te, test_df = load_xy("test")
print("X train:", x_tr.shape, " X test:", x_te.shape)

### 3.2 How we judge a model

Each model gives every test customer a **score**: higher means more likely to default. We
use three measures:

| measure | question it answers | random guess | perfect |
|---|---|---|---|
| **ROC-AUC** | pick one defaulter and one non-defaulter: how often does the defaulter get the higher score? | 0.5 | 1.0 |
| **PR-AUC** | how clean is the top of the list (defaulters vs false alarms)? | the default rate, 0.0046 | 1.0 |
| **recall@5%** | if we review only the 5% highest scores, what share of all defaulters do we find? | 0.05 | 1.0 |

With only 42 defaulters, a result can change a lot depending on which customers happen to be in
the test set. To see how much, we **resample** the test set 1,000 times (draw customers with
replacement), compute the measure each time, and report the middle 95% of the results as a range.

In [ ]:
def recall_at(y, p, frac=0.05):
    k = max(1, int(round(frac * len(y))))
    top = np.argsort(-p, kind="stable")[:k]
    return y[top].sum() / y.sum()


def evaluate(name, y, p, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed)
    roc, pr, rec = [], [], []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        if y[i].sum() == 0:
            continue
        roc.append(roc_auc_score(y[i], p[i]))
        pr.append(average_precision_score(y[i], p[i]))
        rec.append(recall_at(y[i], p[i]))
    k = int(round(0.05 * len(y)))
    caught = int(y[np.argsort(-p, kind="stable")[:k]].sum())

    def fmt(value, sample):
        return f"{value:.3f} ({np.percentile(sample, 2.5):.3f}-{np.percentile(sample, 97.5):.3f})"

    return {"model": name,
            "ROC-AUC (95% range)": fmt(roc_auc_score(y, p), roc),
            "PR-AUC (95% range)": fmt(average_precision_score(y, p), pr),
            "recall@5% (95% range)": fmt(recall_at(y, p), rec),
            "caught in top 5%": f"{caught} of {int(y.sum())}"}


results = []

### 3.3 Logistic regression (LR)

LR learns **one weight per embedding number** plus a constant. The score is
`sigmoid(w1*x1 + ... + w192*x192 + b)`. It can only draw a straight boundary through the
192-number space.

- `StandardScaler` first rescales every column to mean 0 and spread 1, so no column dominates
  because of its scale. It is fitted on the **train** set only.
- `C=1.0` controls how strongly large weights are penalised (smaller C = stronger penalty).

In [ ]:
lr = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=5000))
lr.fit(x_tr, y_tr)
p_lr = lr.predict_proba(x_te)[:, 1]
results.append(evaluate("LR", y_te, p_lr))
display(pd.DataFrame(results))

w = lr[-1].coef_[0]
print("the 5 embedding numbers with the largest weights:",
      [(f"emb_{i}", round(float(w[i]), 3)) for i in np.argsort(-np.abs(w))[:5]])

### 3.4 Gradient boosted trees (GBDT)

GBDT builds many small decision trees, one after another. Each new tree corrects the mistakes of
the trees before it. Trees can find bends and combinations that LR cannot. Scaling is not needed,
because a tree only asks "is this number above or below a threshold?".

| setting | value | why |
|---|---|---|
| `learning_rate` | 0.05 | each tree changes the score only a little, so no single tree takes over |
| `max_iter` | 300 | the number of trees |
| `max_leaf_nodes` | 15 | small trees |
| `min_samples_leaf` | 50 | every leaf needs 50 customers. With only 94 defaulters, smaller leaves would learn single customers by heart |
| `l2_regularization` | 1.0 | another brake on extreme values in the leaves |

In [ ]:
gbdt = HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, max_leaf_nodes=15,
                                      min_samples_leaf=50, l2_regularization=1.0, random_state=0)
gbdt.fit(x_tr, y_tr)
p_gbdt = gbdt.predict_proba(x_te)[:, 1]
results.append(evaluate("GBDT", y_te, p_gbdt))
display(pd.DataFrame(results))

### 3.5 A small neural network (MLP)

The MLP has layers: 192 inputs → 128 → 64 → 1 output, with a ReLU between layers so it can learn
bends. We build and train it by hand in PyTorch so every step is visible.

- **Scaling:** the same `StandardScaler` idea as LR, fitted on train.
- **`pos_weight`:** defaulters are rare, so a mistake on a defaulter counts about 190 times as much
  as a mistake on a non-defaulter. Without this, the network learns to say "no default" for
  everyone.
- **Dropout 0.3:** during training, 30% of the units are switched off at random each step, so the
  network can't depend on a few of them.
- **Early stopping:** we hold out 20% of the train set. After each pass over the data (an
  **epoch**) we measure the loss on it. When it stops improving for 15 epochs, we stop and keep the
  best version.

In [ ]:
def train_mlp(x_train, y_train, x_test, seed=0, max_epochs=200, patience=15):
    torch.manual_seed(seed)
    xa, xv, ya, yv = train_test_split(x_train, y_train, test_size=0.2, stratify=y_train,
                                      random_state=seed)

    def t(a):
        return torch.tensor(a, dtype=torch.float32, device=DEVICE)

    xa, xv, ya, yv, xt = t(xa), t(xv), t(ya), t(yv), t(x_test)

    model = torch.nn.Sequential(
        torch.nn.Linear(xa.shape[1], 128), torch.nn.ReLU(), torch.nn.Dropout(0.3),
        torch.nn.Linear(128, 64), torch.nn.ReLU(), torch.nn.Dropout(0.3),
        torch.nn.Linear(64, 1)).to(DEVICE)
    pos_weight = t([(len(ya) - ya.sum().item()) / ya.sum().item()])
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
    print(f"pos_weight = {pos_weight.item():.0f}")

    gen = torch.Generator().manual_seed(seed)
    best, best_state, waited, history = float("inf"), None, 0, []
    for epoch in range(max_epochs):
        model.train()
        for idx in torch.randperm(len(xa), generator=gen).split(256):
            idx = idx.to(DEVICE)
            opt.zero_grad()
            loss_fn(model(xa[idx]).squeeze(1), ya[idx]).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            v = loss_fn(model(xv).squeeze(1), yv).item()
        history.append(v)
        if v < best - 1e-4:
            best, waited = v, 0
            best_state = {k: p.detach().clone() for k, p in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience:
                break
    print(f"stopped after {epoch + 1} epochs; best held-out loss {best:.4f}")
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(xt).squeeze(1)).cpu().numpy(), history


scaler = StandardScaler().fit(x_tr)
p_mlp, history = train_mlp(scaler.transform(x_tr).astype("float32"), y_tr.astype("float32"),
                           scaler.transform(x_te).astype("float32"))
results.append(evaluate("MLP", y_te, p_mlp))

pd.Series(history, name="held-out loss").plot(xlabel="epoch", title="MLP early stopping")
display(pd.DataFrame(results))

### 3.6 What the top of the list looks like

A bank would act on the highest scores. Here are the 10 test customers each model ranks highest,
with their real label.

In [ ]:
top = pd.DataFrame({"user_id": test_df.user_id, "label": y_te,
                    "LR": p_lr, "GBDT": p_gbdt, "MLP": p_mlp})
for m in ("LR", "GBDT", "MLP"):
    t10 = top.nlargest(10, m)
    print(f"{m:4}: defaulters in its top 10 = {int(t10.label.sum())}")
display(top.nlargest(10, "LR"))

---
## 4. Summary

**The same three models on different embeddings** (test set, 9,032 customers, 42 defaulters).
These come from our full test runs, where MLP was averaged over 5 seeds:

| model | fm_t4 (192) | TCN (256) | TCN custom (256) |
|---|---|---|---|
| LR, ROC-AUC | 0.756 | 0.802 | 0.868 |
| GBDT, ROC-AUC | 0.785 | 0.794 | 0.802 |
| LR, caught in top 5% | 8 of 42 | 11 of 42 | 19 of 42 |
| GBDT, caught in top 5% | 5 of 42 | 15 of 42 | 18 of 42 |

For reference, Kumo working on the 12 raw tables directly reaches ROC-AUC 0.852 and catches
21 of 42 in the top 5%.

**What to take away:**

1. **Training data:** a label is one fact about the future per customer at a cut date. The model
   sees only data from before the cut. Train and test are different customers.
2. **Embeddings:** intake → tokenize (with the model's own tokenizer) → embed gives one fixed
   vector per customer. The same vector can feed any downstream task.
3. **Models:** LR, GBDT and MLP are all cheap to train on embeddings. With so few defaulters, the
   quality of the embedding matters more than the choice of model: the same LR goes from 0.756 to
   0.868 ROC-AUC just by changing the embeddings.
4. **Judging:** with a rare class, ROC-AUC alone hides a lot. Look at PR-AUC and recall@5% too,
   and always report a range.